# Week 5 — restriction-map search tree / prune

Open in VS Code. **Select Kernel** (Python 3) in the top right. Then **Run All** from the top. Needs `matplotlib`.

Same toys as the lecture. Distances $L=\{2,2,3,3,4,5,6,7,8,10\}$. Ends $0$ and $10$. Then the motif alignment: leaf $s=(3,1,2)$ and the partial node `ACGT` / `CACG`.

**What to do**

1. Confirm $X=\{0,2,4,7,10\}$ reproduces $L$. Read the number line.
2. Read the prune: `legal([0,8,10], 7, L)` is `(False, 1)`. Uncomment the play line — does the other child live?
3. Motif bound: $B=1$, $U$ for `ACGT`/`CACG` is $4$. Change the second window to `ACGA` and print $U$. Watch the two alignments.
4. Write Q1–Q3 below, without notes.


In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

INK, BLUE, WARM = "#1c2430", "#2a5a8c", "#b56a3a"
BLUE_BG, WARM_BG, FADED = "#eef3f8", "#f7f3ee", "#c5ccd6"

L = [2, 2, 3, 3, 4, 5, 6, 7, 8, 10]


def pairwise(X):
    d = []
    for i, a in enumerate(X):
        for b in X[:i]:
            d.append(abs(a - b))
    return sorted(d)


def leftover(X, L):
    bag = Counter(L)
    for d in pairwise(X):
        bag[d] -= 1
    return bag


def legal(X, x, L):
    # New distances from x to every point already in X must still sit in leftover L.
    bag = leftover(X, L)
    needed = [abs(x - y) for y in X]
    for d in needed:
        if bag[d] <= 0:
            return False, d
        bag[d] -= 1
    return True, None


def draw_map(X, reject=None, title=""):
    fig, ax = plt.subplots(figsize=(8.2, 1.5))
    ax.plot([0, 10], [0, 0], color=INK, lw=3, solid_capstyle="round")
    for x in X:
        ax.plot(x, 0, "o", ms=13, color=BLUE, zorder=3)
        ax.text(x, 0.42, str(x), ha="center", fontsize=12, color=INK, fontweight="bold")
    xs = sorted(X)
    for a, b in zip(xs, xs[1:]):
        ax.text((a + b) / 2, -0.45, str(b - a), ha="center", fontsize=11, color="#6b7380")
    if reject is not None:
        ax.plot(reject, 0, "x", ms=16, color=WARM, mew=2.4, zorder=4)
        ax.text(reject, 0.42, str(reject), ha="center", fontsize=12, color=WARM, fontweight="bold")
    ax.set_xlim(-0.6, 10.6)
    ax.set_ylim(-0.85, 0.85)
    ax.axis("off")
    if title:
        ax.set_title(title, loc="left", fontsize=12, color=INK, pad=2)
    plt.show()


def draw_dna(seqs, starts, k=4, title=""):
    fig, ax = plt.subplots(figsize=(8.2, 0.62 * len(seqs) + 0.5))
    for r, seq in enumerate(seqs):
        ax.text(-0.85, -r, f"s{r + 1}", ha="right", va="center", fontsize=11,
                fontfamily="monospace", color="#6b7380", fontweight="bold")
        s = starts[r]
        for i, ch in enumerate(seq):
            hot = s is not None and s <= i < s + k
            rect = plt.Rectangle(
                (i - 0.46, -r - 0.38), 0.92, 0.76,
                fc=WARM_BG if hot else "#ffffff",
                ec=WARM if hot else FADED, lw=1.5, zorder=2,
            )
            ax.add_patch(rect)
            ax.text(i, -r, ch, ha="center", va="center", fontsize=11,
                    fontfamily="monospace", fontweight="bold",
                    color=WARM if hot else INK, zorder=3)
        if s is not None:
            ax.text(len(seq) + 0.35, -r, f"start {s}", ha="left", va="center",
                    fontsize=10, color=WARM)
        else:
            ax.text(len(seq) + 0.35, -r, "free", ha="left", va="center",
                    fontsize=10, color="#6b7380")
    ax.set_xlim(-1.4, max(len(s) for s in seqs) + 3.2)
    ax.set_ylim(-len(seqs) + 0.35, 0.55)
    ax.axis("off")
    if title:
        ax.set_title(title, loc="left", fontsize=12, color=INK, pad=2)
    plt.show()


def col_cost(col):
    maj = max(col.count(x) for x in col)
    return len(col) - maj


def score(rows):
    return sum(col_cost([row[c] for row in rows]) for c in range(len(rows[0])))


def draw_alignment(rows, names=None, title=""):
    names = names or [f"s{i + 1}" for i in range(len(rows))]
    k = len(rows[0])
    cons, costs = [], []
    for c in range(k):
        col = [row[c] for row in rows]
        cons.append(Counter(col).most_common(1)[0][0])
        costs.append(col_cost(col))
    n = len(rows)
    fig, ax = plt.subplots(figsize=(1.3 * k + 2.2, 0.7 * n + 1.6))
    def cell(x, y, ch, hot=False, cons_row=False):
        fc, ec, tc = (WARM_BG, WARM, WARM) if hot else (BLUE_BG, BLUE, INK)
        if cons_row:
            fc, ec, tc = "#ffffff", BLUE, BLUE
        ax.add_patch(plt.Rectangle((x - 0.42, y - 0.38), 0.84, 0.76,
                                   fc=fc, ec=ec, lw=1.6))
        ax.text(x, y, ch, ha="center", va="center", fontsize=13,
                fontfamily="monospace", fontweight="bold", color=tc)
    for r, (name, row) in enumerate(zip(names, rows)):
        ax.text(-0.85, -r, name, ha="right", va="center", fontsize=11,
                fontfamily="monospace", color="#6b7380", fontweight="bold")
        for c, ch in enumerate(row):
            cell(c, -r, ch, hot=(ch != cons[c]))
    ax.text(-0.85, -n, "cons", ha="right", va="center", fontsize=10,
            fontfamily="monospace", color=BLUE, fontweight="bold")
    for c, ch in enumerate(cons):
        cell(c, -n, ch, cons_row=True)
    for c, cost in enumerate(costs):
        ax.text(c, -n - 0.85, f"+{cost}", ha="center", fontsize=11,
                color=WARM if cost else "#6b7380", fontweight="bold")
    ax.text(k + 0.15, -n / 2, f"score = {sum(costs)}", ha="left", va="center",
            fontsize=12, color=WARM, fontweight="bold")
    ax.set_xlim(-1.5, k + 2.4)
    ax.set_ylim(-n - 1.3, 0.55)
    ax.set_aspect("equal")
    ax.axis("off")
    if title:
        ax.set_title(title, loc="left", fontsize=12, color=INK, pad=2)
    plt.show()


## Confirm the lecture placement

Neighbour lengths on the line should be a permutation of $2,2,3,3$. Pairwise, the whole bag $L$.


In [ ]:
X = [0, 2, 4, 7, 10]
print("pairwise", pairwise(X))
print("L       ", sorted(L))
print("match   ", pairwise(X) == sorted(L))
draw_map(X, title="X = {0, 2, 4, 7, 10}   ·   neighbours 2, 2, 3, 3")


## One prune

After $\{0,10\}$, both ways of placing $8$ are legal. From $\{0,8,10\}$, placing $7$ is not — it needs distance $1$. The cross is a proof: skip that subtree.


In [ ]:
print("after {0,10}, leftover", dict(leftover([0, 10], L)))
print("place 8 from left ", legal([0, 10], 8, L))
print("place 2 from right", legal([0, 10], 2, L))
print("from {0,8,10} place 7", legal([0, 8, 10], 7, L), "  (missing that distance -> prune)")
print("from {0,8,10} place 3", legal([0, 8, 10], 3, L))

draw_map([0, 8, 10], reject=7, title="from {0, 8, 10} the site 7 needs d = 1  ·  DEAD")

# Play — the other first child. Uncomment:
# print("from {0,2,10} place 7", legal([0, 2, 10], 7, L))
# draw_map([0, 2, 10], reject=7, title="from {0, 2, 10} place 7")


## Consensus bound on a partial alignment

Lecture leaf $s=(3,1,2)$: windows `ACGT`, `ACGA`, `ACGT`. Consensus `ACGT`, score $0+1+0=1$, so $B=1$.

Partial node: only $s_1=3$ and $s_2=0$ (`ACGT` and `CACG`). Per column, take the most frequent letter, then count the others. Remaining sequences cost $0$ in the dream future, so that sum is $U$. If $U>B$, prune.

Then change the second window to `ACGA` and re-run this cell. Does that node still die?


In [ ]:
seqs = ["ATGACGTTTA", "CACGAGGCT", "TTACGTA"]

B = score(["ACGT", "ACGA", "ACGT"])  # leaf s = (3, 1, 2)
U = score(["ACGT", "CACG"])          # partial: s1=3, s2=0
print("B", B, "  U for ACGT/CACG", U, "  prune", U > B)
assert B == 1 and U == 4 and U > B

draw_dna(seqs, [3, 1, 2], title="leaf s = (3, 1, 2)   ·   B = 1")
draw_alignment(["ACGT", "ACGA", "ACGT"], title="complete alignment")

draw_dna(seqs, [3, 0, None], title="partial node s1 = 3, s2 = 0, s3 free   ·   U = 4")
draw_alignment(["ACGT", "CACG"], names=["s1", "s2"], title="partial alignment  ·  prune")

# Play — the other s2 child. Change CACG to ACGA and re-run.


## Questions

Without looking at notes. **Q1–Q2:** today’s idea. **Q3:** a situation we did not drill.

**Q1.** Brute force as a tree: what sits on a node, what on an edge?

A1.

**Q2.** A bound that lets you skip a subtree — in one sentence. When does the whole tree still get explored?

A2.

**Q3.** Week 2’s TSP: a partial tour already costs 40; you hold a complete tour of cost 35. What do you do with that branch, and why is that still exact?

A3.
